In [ ]:
# Importamos las librerías necesarias
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Configuramos el estilo de los gráficos para que se vean mejor
plt.style.use('seaborn-v0_8-darkgrid')

print("Librerías importadas correctamente. ¡Listo para descargar datos!")

In [ ]:
# Definimos el activo (ticker) y las fechas
ticker = "BTC-USD"
fecha_inicio = "2022-01-01"

print(f"Descargando datos de {ticker}...")
# Descargamos los datos históricos
datos = yf.download(ticker, start=fecha_inicio)

# Mostramos un gráfico del precio de Cierre (Close)
plt.figure(figsize=(12, 5))
plt.plot(datos['Close'], label=f'Precio de {ticker}', color='orange')
plt.title(f"Historial de Precio de {ticker}")
plt.xlabel("Fecha")
plt.ylabel("Precio en USD")
plt.legend()
plt.show()

# Mostramos las últimas 5 filas de la tabla de datos
datos.tail()

In [ ]:
# 1. Calculamos el Promedio Móvil Simple de 20 días
datos['SMA_20'] = datos['Close'].rolling(window=20).mean()

# 2. Calculamos el Retorno Diario (porcentaje de cambio respecto al día anterior)
datos['Retorno_Diario'] = datos['Close'].pct_change()

# 3. CREAMOS LA VARIABLE OBJETIVO (TARGET) PARA LA IA
# Comparamos si el precio de cierre de "mañana" (shift(-1)) es mayor al de hoy.
# El resultado es True/False, al multiplicarlo por 1 se convierte en 1 o 0.
datos['Target'] = (datos['Close'].shift(-1) > datos['Close']).astype(int)

# Como usamos promedios de 20 días y buscamos hacia el futuro (shift), 
# se generarán filas vacías (NaN) al principio y al final. Las eliminamos:
datos = datos.dropna()

# Mostramos cómo quedó nuestra tabla enriquecida
print("¡Ingeniería de características completada!")
datos[['Close', 'SMA_20', 'Retorno_Diario', 'Target']].tail(10)

In [ ]:
!pip install xgboost scikit-learn

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

# 1. Definimos cuáles columnas son nuestras Pistas (X) y cuál es la Respuesta (Y)
caracteristicas = ['Close', 'Volume', 'SMA_20', 'Retorno_Diario']
X = datos[caracteristicas] # Lo que la IA mira
y = datos['Target']        # Lo que la IA tiene que predecir

# 2. Dividimos los datos (80% pasado para entrenar, 20% futuro para testear)
split_index = int(len(datos) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Días para entrenar: {len(X_train)} | Días para examinar: {len(X_test)}")

# 3. Inicializamos el modelo XGBoost (El Cerebro Matemático)
modelo_xgb = xgb.XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    random_state=42
)

# 4. ¡ENTRENAMOS AL MODELO! (Aquí es donde ocurre la magia)
modelo_xgb.fit(X_train, y_train)

# 5. Le pedimos al modelo que prediga el 20% de datos de prueba
predicciones = modelo_xgb.predict(X_test)

# 6. Calificamos el examen
precision = accuracy_score(y_test, predicciones)
print("-" * 40)
print(f"Precisión del Modelo XGBoost: {precision * 100:.2f}%")
print("-" * 40)
print("\nReporte detallado de clasificación:")
print(classification_report(y_test, predicciones))

In [ ]:
!pip install groq python-dotenv

In [1]:
import os
from dotenv import load_dotenv
from groq import Groq

# 1. Cargamos tu contraseña de Groq desde el archivo .env de tu proyecto
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

# Verificamos si la llave existe
if api_key:
    print("¡Llave de Groq cargada exitosamente!")
    cliente = Groq(api_key=api_key)
else:
    print("ADVERTENCIA: No se encontró GROQ_API_KEY. Revisa tu archivo .env")

# 2. Creamos una función para que el LLM analice el sentimiento
def analizar_noticia(titular):
    prompt = f"""
    Eres un analista de riesgo financiero experto en criptomonedas.
    Lee este titular y califica el sentimiento del mercado con un número.
    -1 (Pánico/Vender), 0 (Neutral), 1 (Euforia/Comprar).
    
    TITULAR: "{titular}"
    
    Responde ÚNICAMENTE con el número (-1, 0, o 1), nada de texto adicional.
    """
    
    respuesta = cliente.chat.completions.create(
        model="qwen/qwen3.8-27b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    
    return int(respuesta.choices[0].message.content.strip())

# 3. ¡Vamos a probarlo en vivo!
noticia_1 = "La SEC aprueba los primeros ETFs de Bitcoin al contado en Estados Unidos"
noticia_2 = "Estados Unidos anuncia prohibición total de minería de criptomonedas por consumo de energía"

print(f"\nAnalizando Noticia 1: '{noticia_1}'")
score_1 = analizar_noticia(noticia_1)
print(f"Puntaje del LLM: {score_1}")

print(f"\nAnalizando Noticia 2: '{noticia_2}'")
score_2 = analizar_noticia(noticia_2)
print(f"Puntaje del LLM: {score_2}")

¡Llave de Groq cargada exitosamente!

Analizando Noticia 1: 'La SEC aprueba los primeros ETFs de Bitcoin al contado en Estados Unidos'
Puntaje del LLM: 1

Analizando Noticia 2: 'Estados Unidos anuncia prohibición total de minería de criptomonedas por consumo de energía'
Puntaje del LLM: -1


In [2]:
def criterio_de_kelly(probabilidad_exito, ratio_ganancia_perdida=1.0):
    """Fórmula matemática para saber qué porcentaje del capital invertir"""
    p = probabilidad_exito
    q = 1.0 - p
    kelly = p - (q / ratio_ganancia_perdida)
    # Si da negativo, no debemos invertir nada
    return max(0.0, kelly)

def el_arbitro(probabilidad_xgb, titular_noticia):
    print("="*50)
    print("🤖 EL ÁRBITRO DE NOSTRADAMUS")
    print("="*50)
    
    # 1. XGBoost habla
    print(f"📊 XGBoost: Mi probabilidad matemática de éxito es del {probabilidad_xgb * 100}%")
    kelly_base = criterio_de_kelly(probabilidad_xgb)
    print(f"🧮 Criterio Kelly recomienda invertir el {kelly_base * 100:.1f}% de nuestro capital.")
    
    # 2. El LLM lee las noticias
    sentimiento_llm = analizar_noticia(titular_noticia)
    
    print("\n📰 ÚLTIMA HORA:", titular_noticia)
    if sentimiento_llm == 1:
        print("🧠 LLM: El sentimiento es EUFORIA (+1). El contexto apoya a la matemática.")
        decision_final = kelly_base
    elif sentimiento_llm == 0:
        print("🧠 LLM: El sentimiento es NEUTRAL (0). Todo normal, operamos con precaución.")
        decision_final = kelly_base * 0.5  # Invertimos la mitad por precaución
    elif sentimiento_llm == -1:
        print("🧠 LLM: ¡ALERTA! El sentimiento es PÁNICO (-1). Las matemáticas no ven esto.")
        decision_final = 0.0  # Abortamos
        
    print("\n⚖️ DECISIÓN FINAL DEL ÁRBITRO:")
    if decision_final > 0:
        print(f"✅ OPERACIÓN APROBADA. Arriesgaremos el {decision_final * 100:.1f}% de la cuenta.")
    else:
        print("❌ OPERACIÓN RECHAZADA. Se aborta para proteger el capital.")
    print("="*50)

# Probemos dos escenarios distintos:

# Escenario A: XGBoost es optimista (65%) y las noticias son buenas
el_arbitro(probabilidad_xgb=0.65, titular_noticia="El uso de criptomonedas alcanza un máximo histórico global")

print("\n")

# Escenario B: XGBoost sigue optimista (65%) porque el precio ha subido, PERO hay una pésima noticia
el_arbitro(probabilidad_xgb=0.65, titular_noticia="Hackers roban 500 millones de dólares del mayor exchange del mundo")

🤖 EL ÁRBITRO DE NOSTRADAMUS
📊 XGBoost: Mi probabilidad matemática de éxito es del 65.0%
🧮 Criterio Kelly recomienda invertir el 30.0% de nuestro capital.

📰 ÚLTIMA HORA: El uso de criptomonedas alcanza un máximo histórico global
🧠 LLM: El sentimiento es EUFORIA (+1). El contexto apoya a la matemática.

⚖️ DECISIÓN FINAL DEL ÁRBITRO:
✅ OPERACIÓN APROBADA. Arriesgaremos el 30.0% de la cuenta.


🤖 EL ÁRBITRO DE NOSTRADAMUS
📊 XGBoost: Mi probabilidad matemática de éxito es del 65.0%
🧮 Criterio Kelly recomienda invertir el 30.0% de nuestro capital.

📰 ÚLTIMA HORA: Hackers roban 500 millones de dólares del mayor exchange del mundo
🧠 LLM: ¡ALERTA! El sentimiento es PÁNICO (-1). Las matemáticas no ven esto.

⚖️ DECISIÓN FINAL DEL ÁRBITRO:
❌ OPERACIÓN RECHAZADA. Se aborta para proteger el capital.
